# Database-to-DataFrame Pipeline
## PostgreSQL + Pandas Data Ingestion

This notebook demonstrates:
1. **Legacy approach**: psycopg2 + SQLAlchemy
2. **Modern approach**: psycopg3 + SQLAlchemy
3. **Cutting-edge**: ADBC (Apache Arrow Database Connectivity)

All Pandas database functions:
- read_sql
- read_sql_table
- read_sql_query
- to_sql
- Chunked reading
- Dtype mapping
- Parameterized queries
- Date parsing

In [5]:
# Import libraries
import os
import warnings
import pandas as pd
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')

## 1. Configuration

In [6]:
# Database configuration
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "olist_db",
    "user": "postgres",
    "password": "0000"
}

# CSV files path
CSV_PATH = r"C:/Users/Info/Downloads/archive"

# List all CSV files
csv_files = [f for f in os.listdir(CSV_PATH) if f.endswith('.csv')]
print("CSV files found:")
for f in csv_files:
    print(f"  - {f}")

CSV files found:
  - olist_customers_dataset.csv
  - olist_geolocation_dataset.csv
  - olist_orders_dataset.csv
  - olist_order_items_dataset.csv
  - olist_order_payments_dataset.csv
  - olist_order_reviews_dataset.csv
  - olist_products_dataset.csv
  - olist_sellers_dataset.csv
  - product_category_name_translation.csv


## 2. Database Connection Functions

In [7]:
def get_psycopg2_engine():
    """Legacy psycopg2 connection"""
    return create_engine(
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}",
        pool_pre_ping=True
    )

def get_psycopg3_engine():
    """Modern psycopg3 connection"""
    return create_engine(
        f"postgresql+psycopg://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}",
        pool_pre_ping=True
    )

def get_adbc_engine():
    """ADBC (Apache Arrow) connection"""
    return create_engine(
        f"postgresql+adbc://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    )

print("Connection functions defined successfully!")

Connection functions defined successfully!


## 3. Ensure Database Exists

In [8]:
# Connect to default postgres database to create our database if not exists
default_conn_string = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/postgres"
)
engine_default = create_engine(default_conn_string, isolation_level="AUTOCOMMIT")

with engine_default.connect() as conn:
    result = conn.execute(text(
        f"SELECT 1 FROM pg_database WHERE datname = '{DB_CONFIG['database']}'"
    ))
    exists = result.fetchone()
    
    if not exists:
        conn.execute(text(f"CREATE DATABASE {DB_CONFIG['database']}"))
        print(f"Created database: {DB_CONFIG['database']}")
    else:
        print(f"Database already exists: {DB_CONFIG['database']}")

engine_default.dispose()

Database already exists: olist_db


## 4. Load CSV Files into DataFrames

In [9]:
# Load all CSV files
csv_data = {}

for file in csv_files:
    # Get table name from filename
    table_name = file.replace('olist_', '').replace('_dataset.csv', '').replace('_translation.csv', '')
    
    # Load CSV
    df = pd.read_csv(os.path.join(CSV_PATH, file))
    csv_data[table_name] = df
    
    print(f"{table_name}: {len(df):,} rows, {len(df.columns)} columns")

print("\nAll CSV files loaded successfully!")

customers: 99,441 rows, 5 columns
geolocation: 1,000,163 rows, 5 columns
orders: 99,441 rows, 8 columns
order_items: 112,650 rows, 7 columns
order_payments: 103,886 rows, 5 columns
order_reviews: 99,224 rows, 7 columns
products: 32,951 rows, 9 columns
sellers: 3,095 rows, 4 columns
product_category_name: 71 rows, 2 columns

All CSV files loaded successfully!


## 5. Import Data to PostgreSQL (psycopg2 + SQLAlchemy)

In [10]:
# Connect using psycopg2
engine = get_psycopg2_engine()
print("Connected to PostgreSQL using psycopg2")

# Drop existing tables
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS order_items CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS orders CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS order_payments CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS order_reviews CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS customers CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS products CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS sellers CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS geolocation CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS product_category_name CASCADE"))
    conn.commit()

print("Existing tables dropped")

# Import each CSV to database
table_mapping = {
    'customers': 'customers',
    'orders': 'orders',
    'order_items': 'order_items',
    'order_payments': 'order_payments',
    'order_reviews': 'order_reviews',
    'products': 'products',
    'sellers': 'sellers',
    'geolocation': 'geolocation',
    'category_name_translation': 'product_category_name'
}

for table_key, table_name in table_mapping.items():
    if table_key in csv_data:
        df = csv_data[table_key]
        df.to_sql(table_name, engine, if_exists='replace', index=False)
        print(f"Imported {table_name}: {len(df):,} rows")

print("\nAll data imported successfully using psycopg2!")

Connected to PostgreSQL using psycopg2
Existing tables dropped
Imported customers: 99,441 rows
Imported orders: 99,441 rows
Imported order_items: 112,650 rows
Imported order_payments: 103,886 rows
Imported order_reviews: 99,224 rows
Imported products: 32,951 rows
Imported sellers: 3,095 rows
Imported geolocation: 1,000,163 rows

All data imported successfully using psycopg2!


## 6. Pandas Database Functions

### 6.1 read_sql - Generic SQL read

In [11]:
# read_sql - works with both SQL queries and table names
query = "SELECT * FROM customers LIMIT 5"
df_read_sql = pd.read_sql(query, engine)
print("read_sql result:")
print(df_read_sql)

read_sql result:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  


### 6.2 read_sql_query - Read SQL query

In [12]:
# read_sql_query - specifically for SQL queries
query = """
    SELECT customer_state, COUNT(*) as count 
    FROM customers 
    GROUP BY customer_state 
    ORDER BY count DESC 
    LIMIT 10
"""
df_query = pd.read_sql_query(query, engine)
print("read_sql_query result (Top 10 states by customer count):")
print(df_query)

read_sql_query result (Top 10 states by customer count):
  customer_state  count
0             SP  41746
1             RJ  12852
2             MG  11635
3             RS   5466
4             PR   5045
5             SC   3637
6             BA   3380
7             DF   2140
8             ES   2033
9             GO   2020


### 6.3 read_sql_table - Read entire table

In [13]:
# read_sql_table - read entire table directly
df_table = pd.read_sql_table('customers', engine)
print(f"read_sql_table result: {len(df_table):,} rows")
print(df_table.head())

read_sql_table result: 99,441 rows
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  


### 6.4 to_sql - Write DataFrame to database

In [14]:
# Create a sample DataFrame to write
sample_df = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'value': [100.5, 200.75, 300.25]
})

# Write to database
sample_df.to_sql('sample_table', engine, if_exists='replace', index=False)
print("Data written to 'sample_table'")

# Read back
df_back = pd.read_sql('SELECT * FROM sample_table', engine)
print("Read back from database:")
print(df_back)

Data written to 'sample_table'
Read back from database:
   id     name   value
0   1    Alice  100.50
1   2      Bob  200.75
2   3  Charlie  300.25


### 6.5 Chunked Reading - Handle large datasets

In [15]:
# Chunked reading - process large data in batches
chunk_size = 10000
total_rows = 0

print("Processing customers in chunks:")
for chunk in pd.read_sql('SELECT * FROM customers', engine, chunksize=chunk_size):
    total_rows += len(chunk)
    print(f"  Processed chunk: {len(chunk)} rows")

print(f"\nTotal rows processed: {total_rows}")

Processing customers in chunks:
  Processed chunk: 10000 rows
  Processed chunk: 10000 rows
  Processed chunk: 10000 rows
  Processed chunk: 10000 rows
  Processed chunk: 10000 rows
  Processed chunk: 10000 rows
  Processed chunk: 10000 rows
  Processed chunk: 10000 rows
  Processed chunk: 10000 rows
  Processed chunk: 9441 rows

Total rows processed: 99441


### 6.6 Dtype Mapping - Explicit column types

In [16]:
# Dtype mapping - specify column types explicitly
dtype_mapping = {
    'customer_id': 'string',
    'customer_unique_id': 'string',
    'customer_zip_code_prefix': 'string',
    'customer_city': 'string',
    'customer_state': 'string'
}

df_dtype = pd.read_sql('SELECT * FROM customers LIMIT 100', engine, dtype=dtype_mapping)
print("Data types with dtype mapping:")
print(df_dtype.dtypes)

Data types with dtype mapping:
customer_id                 string
customer_unique_id          string
customer_zip_code_prefix    string
customer_city               string
customer_state              string
dtype: object


### 6.7 Parameterized Queries - Prevent SQL injection

In [17]:
# Parameterized queries - safe query execution
# Get some customer IDs
customer_ids = pd.read_sql('SELECT customer_id FROM customers LIMIT 5', engine)['customer_id'].tolist()

# Use parameterized query
query = "SELECT * FROM customers WHERE customer_id = ANY(:ids)"
df_param = pd.read_sql(text(query), engine, params={'ids': customer_ids})
print(f"Parameterized query result: {len(df_param)} rows")
print(df_param)

Parameterized query result: 5 rows
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  


### 6.8 Date Parsing - Parse datetime columns

In [18]:
# Date parsing - convert string dates to datetime
parse_dates = ['order_purchase_timestamp', 'order_approved_at', 
               'order_delivered_carrier_date', 'order_delivered_customer_date',
               'order_estimated_delivery_date']

df_orders = pd.read_sql('SELECT * FROM orders LIMIT 10', engine, parse_dates=parse_dates)
print("Orders with parsed dates:")
print(df_orders.dtypes)
print("\nSample data:")
print(df_orders.head())

Orders with parsed dates:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Sample data:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15 

## 7. Modern Approach: psycopg3 + SQLAlchemy

In [19]:
# Connect using psycopg3
try:
    engine_psycopg3 = get_psycopg3_engine()
    df_psycopg3 = pd.read_sql('SELECT * FROM customers LIMIT 5', engine_psycopg3)
    print("Successfully connected using psycopg3!")
    print(df_psycopg3)
except Exception as e:
    print(f"psycopg3 connection failed: {e}")

Successfully connected using psycopg3!
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  


## 8. Cutting-edge: ADBC (Apache Arrow Database Connectivity)

In [20]:
# ADBC - Apache Arrow Database Connectivity for 5-10x faster data transfer
try:
    engine_adbc = get_adbc_engine()
    df_adbc = pd.read_sql('SELECT * FROM customers LIMIT 5', engine_adbc)
    print("Successfully connected using ADBC!")
    print(df_adbc)
except Exception as e:
    print(f"ADBC connection failed: {e}")
    print("\nTo install ADBC driver: uv pip install adbc-driver-postgresql")

ADBC connection failed: Can't load plugin: sqlalchemy.dialects:postgresql.adbc

To install ADBC driver: uv pip install adbc-driver-postgresql


## 9. Performance Comparison

In [21]:
import time

# Test read performance with different connectors
connectors = [
    ("psycopg2", get_psycopg2_engine),
    ("psycopg3", get_psycopg3_engine),
]

results = {}

for name, get_engine in connectors:
    try:
        eng = get_engine()
        
        # Read test
        start = time.time()
        df = pd.read_sql('SELECT * FROM customers', eng)
        read_time = time.time() - start
        
        results[name] = {
            'read_time': read_time,
            'rows': len(df)
        }
        
        print(f"{name}: Read {len(df):,} rows in {read_time:.3f}s")
        
        eng.dispose()
    except Exception as e:
        print(f"{name} failed: {e}")

print("\nPerformance comparison complete!")

psycopg2: Read 99,441 rows in 1.378s
psycopg3: Read 99,441 rows in 1.163s

Performance comparison complete!


## 10. Query Examples

In [22]:
# Example: Join customers with orders
query = """
    SELECT 
        c.customer_id,
        c.customer_city,
        c.customer_state,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    LIMIT 10
"""

df_joined = pd.read_sql(query, engine)
print("Customers joined with Orders:")
print(df_joined)

Customers joined with Orders:
                        customer_id   customer_city customer_state  \
0  9ef432eb6251297304e76186b10a928d       sao paulo             SP   
1  8ab97904e6daea8866dbdbc4fb7aad2c     santo andre             SP   
2  503740e9ca751ccdda7ba28e9ab8f608    congonhinhas             PR   
3  f54a9f0e6b351c431402b8461ea51999    faxinalzinho             RS   
4  d3e3b74c766bc6214e0c830b17ee2341      ouro preto             MG   
5  19402a48fe860416adf93348aba37740       sao paulo             SP   
6  3df704f53d3f1d4818840b34ec672a9f       sao paulo             SP   
7  3b6828a50ffe546942b7a473d70ac0fc         goiania             GO   
8  738b086814c6fcc74b8cc583f8516ee3  rio de janeiro             RJ   
9  059f7fc5719c7da6cbafe370971a8d70     hortolandia             SP   

                           order_id order_status order_purchase_timestamp  
0  e481f51cbdc54678b7cc49136f2d6af7    delivered      2017-10-02 10:56:33  
1  ad21c59c0840e6cb83a9ceb5573f8159    delivere

In [23]:
# Example: Aggregate query - Orders by status
query = """
    SELECT 
        order_status,
        COUNT(*) as total_orders
    FROM orders
    GROUP BY order_status
    ORDER BY total_orders DESC
"""

df_status = pd.read_sql(query, engine)
print("Orders by status:")
print(df_status)

Orders by status:
  order_status  total_orders
0    delivered         96478
1      shipped          1107
2     canceled           625
3  unavailable           609
4     invoiced           314
5   processing           301
6      created             5
7     approved             2


In [24]:
# Example: Date-based query
query = """
    SELECT 
        DATE(order_purchase_timestamp) as date,
        COUNT(*) as orders
    FROM orders
    WHERE order_purchase_timestamp >= '2018-01-01'
    GROUP BY DATE(order_purchase_timestamp)
    ORDER BY date
    LIMIT 10
"""

df_date = pd.read_sql(query, engine)
print("Orders by date (from 2018):")
print(df_date)

Orders by date (from 2018):
         date  orders
0  2018-01-01      74
1  2018-01-02     204
2  2018-01-03     225
3  2018-01-04     258
4  2018-01-05     210
5  2018-01-06     216
6  2018-01-07     196
7  2018-01-08     293
8  2018-01-09     252
9  2018-01-10     277


## 11. Close Connection

In [25]:
# Close the main engine connection
engine.dispose()
print("Database connection closed.")

print("\n" + "="*60)
print("Database-to-DataFrame Pipeline Complete!")
print("="*60)

Database connection closed.

Database-to-DataFrame Pipeline Complete!
